# 0 · Data caches & router training

Builds the per-image feature/loss caches (the frozen detector forwarded along the
BASE and SUPER paths) and trains the TinyConv advantage-regression router on them.
Everything downstream consumes these caches / router checkpoints.

- `build_cache.py` — dumps `input/pred` feature grids + `loss_base/loss_super` per image.
- `train_policy.py` — regresses the advantage A = L_base − L_super; selects the
  checkpoint with min validation MSE (`--select val_mse`). 5 seeds.

In [ ]:
# --- setup: run from the repo root with the kernel's own python ---
import os, sys
while not os.path.isdir(f'{os.getcwd()}/method02_advantage_regress_tinyConv'):
    os.chdir('..')          # walk up to the anydepth-yolov12 repo root
PY = sys.executable
print('cwd =', os.getcwd()); print('python =', PY)

## Build caches (KITTI / BDD100K)
Skip if `outputs/<ds>/cache_{train,val}_g2.pt` already exist.

In [ ]:
# KITTI cache (grid 2x2). Heavy: forwards the detector over all images, both paths.
!{PY} -m method02_advantage_regress_tinyConv.build_cache --help

## Train the router (5 seeds, min-val-MSE checkpoint)

In [ ]:
# Example (BDD100K, feat=input). See run_*.sh for the exact sweeps used.
!{PY} -m method02_advantage_regress_tinyConv.train_policy --dataset bdd100k --feat input --norm batch \
    --cache method02_advantage_regress_tinyConv/outputs/bdd100k/cache_train_g2.pt --val_cache method02_advantage_regress_tinyConv/outputs/bdd100k/cache_val_g2.pt \
    --epochs 30 --batch 256 --lr 1e-3 --select val_mse \
    --out method02_advantage_regress_tinyConv/outputs/bdd100k/policy_scenario_s0.pt